<a href="https://colab.research.google.com/github/MuhammadAmaan178/Demand-Lens/blob/main/notebooks/DLens_10_Backend_Data_Export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Notebook 10: Backend Data Export (Store CA_1)**

## Step 1: Mount Drive and Check Files

We connect Google Drive and set our folders inside `DemandLens/Data`. Then we check where each input file sits so we can load everything cleanly.

In [1]:
import numpy as np
import pandas as pd

In [2]:
import os
import json
import sys
import importlib
import joblib
from google.colab import drive

In [3]:
drive.mount("/content/drive")

BASE_DATA = "/content/drive/MyDrive/DemandLens/Data"
PROCESSED_DIR = os.path.join(BASE_DATA, "processed")
PREPROCESSED_DIR = os.path.join(BASE_DATA, "preprocessed")
MODELS_OUT = os.path.join(BASE_DATA, "models")
DATA_OUT = os.path.join(BASE_DATA, "data")
CODE_OUT = "/content/drive/MyDrive/DemandLens/backend_code"

os.makedirs(MODELS_OUT, exist_ok=True)
os.makedirs(DATA_OUT, exist_ok=True)
os.makedirs(CODE_OUT, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
def find_file(fname):
    for folder in [PROCESSED_DIR, PREPROCESSED_DIR, MODELS_OUT, DATA_OUT, CODE_OUT]:
        path = os.path.join(folder, fname)
        if os.path.exists(path):
            return path
    return os.path.join(PROCESSED_DIR, fname)

needed_files = [
    "CA_1_clean.parquet",
    "calendar.csv",
    "feature_lists.json",
    "ca1_h7_val.parquet",
    "ca1_h28_val.parquet",
    "lightgbm_h7.joblib",
    "lightgbm_h28.joblib",
    "anomalies_validation.parquet",
    "item_groups.csv",
    "group_thresholds.csv",
    "item_error_spread.csv",
    "reorder.py"
]

for fname in needed_files:
    path = find_file(fname)
    print(f"[{'OK' if os.path.exists(path) else 'MISSING'}] {fname} -> {path}")

[OK] CA_1_clean.parquet -> /content/drive/MyDrive/DemandLens/Data/processed/CA_1_clean.parquet
[OK] calendar.csv -> /content/drive/MyDrive/DemandLens/Data/preprocessed/calendar.csv
[OK] feature_lists.json -> /content/drive/MyDrive/DemandLens/Data/processed/feature_lists.json
[OK] ca1_h7_val.parquet -> /content/drive/MyDrive/DemandLens/Data/processed/ca1_h7_val.parquet
[OK] ca1_h28_val.parquet -> /content/drive/MyDrive/DemandLens/Data/processed/ca1_h28_val.parquet
[OK] lightgbm_h7.joblib -> /content/drive/MyDrive/DemandLens/Data/processed/lightgbm_h7.joblib
[OK] lightgbm_h28.joblib -> /content/drive/MyDrive/DemandLens/Data/processed/lightgbm_h28.joblib
[OK] anomalies_validation.parquet -> /content/drive/MyDrive/DemandLens/Data/processed/anomalies_validation.parquet
[OK] item_groups.csv -> /content/drive/MyDrive/DemandLens/Data/processed/item_groups.csv
[OK] group_thresholds.csv -> /content/drive/MyDrive/DemandLens/Data/processed/group_thresholds.csv
[OK] item_error_spread.csv -> /conten

## Step 2: Load Data and Feature Lists

We load the cleaned sales data, calendar table, and `feature_lists.json` straight from Drive and check their sizes and date ranges.

In [5]:
ca1_clean = pd.read_parquet(find_file("CA_1_clean.parquet"))
calendar_df = pd.read_csv(find_file("calendar.csv"))

with open(find_file("feature_lists.json"), "r") as f:
    feature_lists = json.load(f)

print("CA_1_clean:", ca1_clean.shape, "| Max date:", pd.to_datetime(ca1_clean["date"]).max().date())
print("Calendar:  ", calendar_df.shape, "| Dates:", calendar_df["date"].min(), "to", calendar_df["date"].max())
print("Features -> H7:", len(feature_lists["features_7"]), "| H28:", len(feature_lists["features_28"]), "| Items:", len(feature_lists["item_categories"]))

CA_1_clean: (4781564, 13) | Max date: 2016-05-22
Calendar:   (1969, 14) | Dates: 2011-01-29 to 2016-06-19
Features -> H7: 27 | H28: 27 | Items: 3049


## Step 3: Save `history_small.parquet` to Drive

We keep data up to `2016-04-24` so test dates stay untouched, take the last 200 days per item, and write `history_small.parquet` directly into our Drive `data` folder.

In [6]:
ca1_clean["date"] = pd.to_datetime(ca1_clean["date"])
cutoff = pd.Timestamp("2016-04-24")

history_small = (
    ca1_clean[ca1_clean["date"] <= cutoff]
    .sort_values(["item_id", "date"])
    .groupby("item_id", sort=False, observed=False)
    .tail(200)[["item_id", "dept_id", "cat_id", "store_id", "date", "sales", "sell_price"]]
    .reset_index(drop=True)
)

hist_out_path = os.path.join(DATA_OUT, "history_small.parquet")
history_small.to_parquet(hist_out_path, index=False)

print("Saved:", hist_out_path, "| Shape:", history_small.shape, "| Max date:", history_small["date"].max().date())

Saved: /content/drive/MyDrive/DemandLens/Data/data/history_small.parquet | Shape: (609372, 7) | Max date: 2016-04-24


## Step 4: Save `calendar_small.csv` to Drive

We pick the calendar columns needed for store `CA_1` and save `calendar_small.csv` straight into the Drive `data` folder.

In [7]:
cal_cols = ["date", "weekday", "wday", "month", "year", "event_name_1", "event_type_1", "snap_CA"]
calendar_small = calendar_df[cal_cols].sort_values("date").reset_index(drop=True)

cal_out_path = os.path.join(DATA_OUT, "calendar_small.csv")
calendar_small.to_csv(cal_out_path, index=False)

print("Saved:", cal_out_path, "| Shape:", calendar_small.shape)

Saved: /content/drive/MyDrive/DemandLens/Data/data/calendar_small.csv | Shape: (1969, 8)


## Step 5: Check Lag and Rolling Windows

We test shifted lags and rolling averages for one item against the validation files to make sure our shift windows line up.

In [8]:
val_h7 = pd.read_parquet(find_file("ca1_h7_val.parquet"))
val_h28 = pd.read_parquet(find_file("ca1_h28_val.parquet"))

sample_item = "FOODS_1_001"
item_hist = history_small[history_small["item_id"] == sample_item].sort_values("date").copy()
item_hist["lag_7"] = item_hist["sales"].shift(7)
item_hist["roll_7"] = item_hist["sales"].shift(7).rolling(7).mean()
item_hist["lag_28"] = item_hist["sales"].shift(28)
item_hist["roll_28"] = item_hist["sales"].shift(28).rolling(28).mean()

sample_dates = val_h7[val_h7["item_id"] == sample_item].sort_values("date").head(3)["date"]
print(item_hist[item_hist["date"].isin(sample_dates)][["date", "sales", "lag_7", "roll_7", "lag_28", "roll_28"]])

          date  sales  lag_7    roll_7  lag_28   roll_28
172 2016-03-28      2    0.0  1.000000     1.0  0.750000
173 2016-03-29      1    2.0  1.285714     0.0  0.678571
174 2016-03-30      1    2.0  1.571429     1.0  0.714286


## Step 6: Write and Test `features.py`

We save `features.py` directly on Drive with `build_features()`, using string grouping so filtering to a few items does not add unused categories. Then we check it against the validation sets and test future dates.

In [9]:
features_script = """import numpy as np
import pandas as pd

EVENT_TYPES = ["Cultural", "National", "Religious", "Sporting", "None"]
DEPARTMENTS = [
    "FOODS_1", "FOODS_2", "FOODS_3",
    "HOBBIES_1", "HOBBIES_2",
    "HOUSEHOLD_1", "HOUSEHOLD_2"
]

def build_features(history_df, calendar_df, feature_lists, horizon=7, future_dates=None):
    if horizon not in (7, 28):
        raise ValueError("horizon must be 7 or 28")

    df = history_df[["item_id", "dept_id", "date", "sales", "sell_price"]].copy()
    df["item_id"] = df["item_id"].astype(str)
    df["date"] = pd.to_datetime(df["date"])

    if future_dates is not None and len(future_dates) > 0:
        future_dates = pd.to_datetime(future_dates)
        last_rows = (
            df.sort_values(["item_id", "date"])
            .groupby("item_id", as_index=False)
            .last()[["item_id", "dept_id", "sell_price"]]
        )
        future_rows = []
        for d in future_dates:
            row = last_rows.copy()
            row["date"] = d
            row["sales"] = np.nan
            future_rows.append(row)
        df = pd.concat([df] + future_rows, ignore_index=True)

    df = df.sort_values(["item_id", "date"]).reset_index(drop=True)

    sales_grp = df.groupby("item_id", sort=False)["sales"]
    if horizon == 7:
        df["lag_7"] = sales_grp.shift(7)
        df["rolling_mean_7"] = sales_grp.transform(lambda s: s.shift(7).rolling(7).mean())
        cols = feature_lists["features_7"]
    else:
        df["lag_28"] = sales_grp.shift(28)
        df["rolling_mean_28"] = sales_grp.transform(lambda s: s.shift(28).rolling(28).mean())
        cols = feature_lists["features_28"]

    df["price_change"] = df.groupby("item_id", sort=False)["sell_price"].transform(
        lambda s: s - s.shift(7)
    ).fillna(0.0)

    cal = calendar_df[["date", "wday", "month", "year", "event_type_1", "snap_CA"]].copy()
    cal["date"] = pd.to_datetime(cal["date"])
    df = df.merge(cal, on="date", how="left")

    df["snap"] = df["snap_CA"].fillna(0).astype(int)
    df["is_weekend"] = df["wday"].isin([1, 2]).astype(int)
    df["has_event"] = df["event_type_1"].notna().astype(int)

    ev_clean = df["event_type_1"].fillna("None")
    for ev in EVENT_TYPES:
        df[f"ev_{ev}"] = (ev_clean == ev).astype(int)

    for dept in DEPARTMENTS:
        df[f"dept_{dept}"] = (df["dept_id"] == dept).astype(int)

    df["wday_sin"] = np.sin(2 * np.pi * df["wday"] / 7)
    df["wday_cos"] = np.cos(2 * np.pi * df["wday"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["item_id"] = pd.Categorical(df["item_id"], categories=feature_lists["item_categories"])

    if future_dates is not None and len(future_dates) > 0:
        df = df[df["date"].isin(future_dates)].copy()

    return df[["date"] + cols].reset_index(drop=True)
"""

features_path = os.path.join(CODE_OUT, "features.py")
with open(features_path, "w") as f:
    f.write(features_script)

if CODE_OUT not in sys.path:
    sys.path.insert(0, CODE_OUT)

import features
importlib.reload(features)
from features import build_features

test_items = ["FOODS_1_001", "FOODS_1_098"]
sub_hist = history_small[history_small["item_id"].isin(test_items)].copy()

out_h7 = build_features(sub_hist, calendar_small, feature_lists, horizon=7)
out_h28 = build_features(sub_hist, calendar_small, feature_lists, horizon=28)

ref_h7 = val_h7[val_h7["item_id"].isin(test_items)].sort_values(["item_id", "date"]).reset_index(drop=True)
cmp_h7 = out_h7[out_h7["date"].isin(ref_h7["date"])].sort_values(["item_id", "date"]).reset_index(drop=True)
max_err_7 = (ref_h7[["lag_7", "rolling_mean_7", "price_change"]].astype(float) - cmp_h7[["lag_7", "rolling_mean_7", "price_change"]].astype(float)).abs().max().max()

future_test = build_features(sub_hist, calendar_small, feature_lists, horizon=7, future_dates=pd.date_range("2016-04-25", periods=7))

print("H7 columns ok:", list(out_h7.columns[1:]) == feature_lists["features_7"], "| Max diff:", round(float(max_err_7), 6))
print("H28 columns ok:", list(out_h28.columns[1:]) == feature_lists["features_28"])
print("Future rows:", len(future_test), "| Null lags:", int(future_test["lag_7"].isna().sum()))

H7 columns ok: True | Max diff: 0.0
H28 columns ok: True
Future rows: 14 | Null lags: 0


## Step 7: Organize Drive Folders and Run Final Check

We put the model files into `models`, the data files into `data`, and the scripts into `backend_code` right on Drive, then run a quick check for missing files, date leakage, and sample predictions.

In [10]:
import shutil

model_files = ["lightgbm_h7.joblib", "lightgbm_h28.joblib"]
data_files = [
    "feature_lists.json",
    "history_small.parquet",
    "calendar_small.csv",
    "anomalies_validation.parquet",
    "item_groups.csv",
    "item_error_spread.csv",
    "group_thresholds.csv"
]
code_files = ["features.py", "reorder.py"]

for fname in model_files:
    src = find_file(fname)
    dst = os.path.join(MODELS_OUT, fname)
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy2(src, dst)

for fname in data_files:
    src = find_file(fname)
    dst = os.path.join(DATA_OUT, fname)
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy2(src, dst)

for fname in code_files:
    src = find_file(fname)
    dst = os.path.join(CODE_OUT, fname)
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy2(src, dst)

missing_models = [f for f in model_files if not os.path.exists(os.path.join(MODELS_OUT, f))]
missing_data = [f for f in data_files if not os.path.exists(os.path.join(DATA_OUT, f))]
missing_code = [f for f in code_files if not os.path.exists(os.path.join(CODE_OUT, f))]
all_missing = missing_models + missing_data + missing_code

saved_hist = pd.read_parquet(os.path.join(DATA_OUT, "history_small.parquet"))
saved_cal = pd.read_csv(os.path.join(DATA_OUT, "calendar_small.csv"))
max_date = pd.to_datetime(saved_hist["date"]).max()
no_leak = max_date <= pd.Timestamp("2016-04-24")

m7 = joblib.load(os.path.join(MODELS_OUT, "lightgbm_h7.joblib"))
m28 = joblib.load(os.path.join(MODELS_OUT, "lightgbm_h28.joblib"))

one_item = saved_hist[saved_hist["item_id"] == "FOODS_1_001"].copy()
f7 = build_features(one_item, saved_cal, feature_lists, horizon=7, future_dates=pd.date_range("2016-04-25", periods=7))
f28 = build_features(one_item, saved_cal, feature_lists, horizon=28, future_dates=pd.date_range("2016-04-25", periods=28))

p7 = np.clip(m7.predict(f7[feature_lists["features_7"]]), 0, None)
p28 = np.clip(m28.predict(f28[feature_lists["features_28"]]), 0, None)

print("Models in Drive:", sorted(os.listdir(MODELS_OUT)))
print("Data in Drive:  ", sorted(os.listdir(DATA_OUT)))
print("Code in Drive:  ", sorted(os.listdir(CODE_OUT)))
print("Missing files:  ", all_missing)
print("Max hist date:  ", max_date.date(), "| No test leakage:", no_leak)
print("Sample forecast -> H7 mean:", round(float(p7.mean()), 2), "| H28 mean:", round(float(p28.mean()), 2))

if not all_missing and no_leak and len(p7) == 7 and len(p28) == 28:
    print("ALL CHECKS PASSED")
else:
    print("CHECK FAILED")

Models in Drive: ['lightgbm_h28.joblib', 'lightgbm_h7.joblib']
Data in Drive:   ['anomalies_validation.parquet', 'calendar_small.csv', 'feature_lists.json', 'group_thresholds.csv', 'history_small.parquet', 'item_error_spread.csv', 'item_groups.csv']
Code in Drive:   ['__pycache__', 'features.py', 'reorder.py']
Missing files:   []
Max hist date:   2016-04-24 | No test leakage: True
Sample forecast -> H7 mean: 0.98 | H28 mean: 0.8
ALL CHECKS PASSED
